# Large coherent cat benchmark

**Publication copy.** The only changed Python cell is the first configuration cell: it finds `data/multimode_cat_plots` locally instead of scanning Google Drive. Numerical reconstruction and comparison cells remain unchanged. Precomputed BI, CO and CGAN arrays are required; they are **not distributed**. See `../docs/DATA.md`. Sections 12–15 are optional historical/bootstrap diagnostics beyond the article comparison.


# Coauthor synthetic benchmark — Stage 2 / final observable comparison

This notebook uses the already-saved BI / CO / NN(CGAN) reconstructions for the
three-component coherent cat state with

\[
a=3+0.5i,\qquad N_\theta=72,\qquad n=2000
\]

samples per original phase setting.

It performs four separate tasks:

1. **Audit the old comparison exactly in the coauthors' position-kernel convention.**
2. Convert the saved BI / CO / NN(CGAN) outputs into a common physical Fock representation
   by first reproducing the position kernels actually compared by the coauthors.
3. Rerun **only the current frozen KQSE estimator** (approximate bandwidth, floor \(0.03\),
   clipping \(0.01\le h\le0.5\)) on the *same* `cat_state_data.dat`.
4. Compare all methods using the exact analytical Wigner function and exact displaced
   photon-number tomograms \(w_0,\ldots,w_3\).

No BI, CO, or CGAN training/optimization is rerun.

The old `KQSE.npy` is retained only for audit of the historical comparison; it is **not**
used as the current KQSE result because Stage 1 showed that it is substantially non-Hermitian.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.special import eval_genlaguerre, eval_hermite, gammaln
from scipy.stats import gaussian_kde

# Publication-copy path adaptation only (no estimator changes).
# Start Jupyter from the repository root, or from notebooks/.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "notebooks").exists():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / "data" / "multimode_cat_plots"
if not (ROOT / "cat_state_data.dat").is_file():
    raise FileNotFoundError(
        "Missing data/multimode_cat_plots/cat_state_data.dat. "
        "See docs/DATA.md for local-only preparation."
    )
PLOTS = ROOT / "plots"

OUTDIR = ROOT / "new_observable_comparison"
OUTDIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUTDIR:", OUTDIR)

SEED = 0
np.random.seed(SEED)

A_CAT = 3.0 + 0.5j
R_LIST = [0,1,2,3]

PHI_ABS_FLOOR = 0.03
H_MIN = 0.01
H_MAX = 0.5

PHI_R = 8.0
PHI_N = 81
N_PRECISION = 1200

Q_GRID = np.linspace(-6.0, 6.0, 81)
P_GRID = np.linspace(-6.0, 6.0, 81)
ALPHA_GRID = np.linspace(-6.0, 6.0, 121).astype(complex)

X_POS = np.linspace(-8.0, 8.0, 401)

In [ ]:
DATA_PATH = ROOT / "cat_state_data.dat"

PATHS = {
    "BI10": PLOTS / "BI_N=10.npy",
    "BI20": PLOTS / "BI_N=20.npy",
    "CO10": PLOTS / "CO_N=10.npy",
    "CO20": PLOTS / "CO_N=20.npy",
    "NN10": PLOTS / "NN_N=10.npy",
    "NN20": PLOTS / "NN_N=20.npy",
    "KQSE_OLD": PLOTS / "KQSE.npy",
}

for k,p in PATHS.items():
    if not p.exists():
        raise FileNotFoundError(p)
    print(k, "->", p)

theta_data, x_data = np.loadtxt(DATA_PATH, unpack=True)

RAW = {
    k: np.load(p, allow_pickle=True)
    for k,p in PATHS.items()
}

print("\nData shape:", theta_data.shape, x_data.shape)
print("Total measurements:", len(x_data))
print("Unique original phase values:", len(np.unique(theta_data)))

for k,a in RAW.items():
    print(k, np.asarray(a).shape, np.asarray(a).dtype)

## 2. Exact three-component cat-state references

The exact target is used here, unlike the experimental kitten benchmark.

In [ ]:
def cat_amplitudes(a=A_CAT):
    return np.array([
        a*np.exp(2j*np.pi*l/3.0)
        for l in range(3)
    ], dtype=complex)

def cat_norm(a=A_CAT):
    amps = cat_amplitudes(a)
    s = 0.0 + 0.0j
    for aj in amps:
        for ak in amps:
            s += np.exp(
                -0.5*abs(aj)**2
                -0.5*abs(ak)**2
                +np.conj(ak)*aj
            )
    return complex(1.0/np.sqrt(s))

def coherent_wavefunction(a, x):
    x = np.asarray(x, float)
    return (
        np.pi**(-0.25)
        * np.exp(
            -x**2/2
            -abs(a)**2/2
            +np.sqrt(2)*a*x
            -a**2/2
        )
    )

def exact_cat_position_kernel_standard(x_grid, a=A_CAT):
    """
    Standard matrix orientation:
        K[i,j] = rho(x_i, x_j).
    """
    x = np.asarray(x_grid, float)
    psi = np.zeros_like(x, dtype=complex)

    for amp in cat_amplitudes(a):
        psi += coherent_wavefunction(amp, x)

    psi *= cat_norm(a)

    return np.outer(
        psi,
        np.conj(psi)
    )

def exact_cat_position_kernel_source_array(x_grid, a=A_CAT):
    """
    Array orientation used by the coauthors' plots.ipynb.

    Their code calls:
        Y, YP = np.meshgrid(y_array, y_array)
    with NumPy's default indexing='xy', and then evaluates
        psi(Y) * conj(psi(YP)).

    Therefore the stored/displayed exact array is the transpose
    of the standard kernel matrix:
        K_source[row(y'), col(y)] = rho(y, y').
    """
    return exact_cat_position_kernel_standard(
        x_grid, a
    ).T

def exact_cat_wigner(q_grid, p_grid, a=A_CAT):
    amps = cat_amplitudes(a)
    Nc = cat_norm(a)

    Q, P = np.meshgrid(
        np.asarray(q_grid,float),
        np.asarray(p_grid,float),
        indexing="ij"
    )
    gamma = (Q + 1j*P)/np.sqrt(2.0)

    total = np.zeros_like(gamma, dtype=complex)

    for aj in amps:
        for ak in amps:
            total += np.exp(
                -2*np.abs(gamma)**2
                +2*gamma*np.conj(ak)
                +2*np.conj(gamma)*aj
                -aj*np.conj(ak)
                -0.5*(abs(aj)**2+abs(ak)**2)
            )

    return np.real_if_close(
        abs(Nc)**2/np.pi * total
    )

def exact_cat_photon(alpha_grid, r, a=A_CAT):
    alpha_grid = np.asarray(alpha_grid, complex)
    amps = cat_amplitudes(a)
    Nc = cat_norm(a)

    out = np.empty(len(alpha_grid), float)

    for i, alpha in enumerate(alpha_grid):
        amp_sum = np.sum(
            np.exp(-np.conj(alpha)*amps)
            * (alpha+amps)**r
        )

        out[i] = (
            abs(Nc)**2
            / math.factorial(r)
            * np.exp(-abs(a)**2-abs(alpha)**2)
            * abs(amp_sum)**2
        )

    return out

# Two explicitly distinguished position-kernel orientations.
RHO_X_EXACT_STANDARD = exact_cat_position_kernel_standard(X_POS)
RHO_X_EXACT_SOURCE = exact_cat_position_kernel_source_array(X_POS)

W_EXACT = np.real(exact_cat_wigner(Q_GRID, P_GRID))
w_EXACT = {
    r: exact_cat_photon(ALPHA_GRID, r)
    for r in R_LIST
}

print("Exact references built.")
print(
    "Standard-kernel trace:",
    np.trapezoid(
        np.real(np.diag(RHO_X_EXACT_STANDARD)),
        X_POS
    )
)

print(
    "Source-array transpose check:",
    np.max(
        np.abs(
            RHO_X_EXACT_SOURCE
            - RHO_X_EXACT_STANDARD.T
        )
    )
)

## 3. Reproduce the coauthors' old position-kernel comparison

The supplied plotting source uses the saved BI/CO/NN outputs with its own
position-space transforms. We reproduce those operations literally first,
before converting anything to Wigner or photon-number tomograms.

In [ ]:
def ho_basis_matrix(x_grid, dim):
    x = np.asarray(x_grid, float)
    Psi = np.empty((dim, len(x)), float)

    for n in range(dim):
        norm = (
            np.pi**(-0.25)
            * np.exp(
                -0.5*(
                    n*np.log(2.0)
                    +gammaln(n+1)
                )
            )
        )
        Psi[n] = (
            norm
            * eval_hermite(n, x)
            * np.exp(-x**2/2)
        )

    return Psi

def position_kernel_from_fock(rho_fock, x_grid):
    """
    Standard orientation:
        K[i,j] = rho(x_i, x_j).
    """
    rho_fock = np.asarray(rho_fock, complex)
    Psi = ho_basis_matrix(x_grid, rho_fock.shape[0])
    return Psi.T @ rho_fock @ Psi

def old_position_error(K, Ktrue):
    return float(
        np.max(
            np.abs(
                np.asarray(K)-np.asarray(Ktrue)
            )**2
        )
    )

# These reproduce the exact arrays constructed in the coauthors'
# plots.ipynb before the old pointwise error is evaluated.
POSITION_REP_SOURCE = {
    "BI10": position_kernel_from_fock(
        RAW["BI10"], X_POS
    ).conj(),

    "BI20": position_kernel_from_fock(
        RAW["BI20"], X_POS
    ).conj(),

    "CO10": np.asarray(RAW["CO10"], complex),
    "CO20": np.asarray(RAW["CO20"], complex),

    "NN10": position_kernel_from_fock(
        RAW["NN10"], X_POS
    ).conj().T[::-1, ::-1],

    "NN20": position_kernel_from_fock(
        RAW["NN20"], X_POS
    ).conj().T[::-1, ::-1],

    "KQSE_OLD": np.asarray(RAW["KQSE_OLD"], complex),
}

old_rows=[]

for label in [
    "KQSE_OLD",
    "BI10","CO10","NN10",
    "BI20","CO20","NN20"
]:
    old_rows.append({
        "result": label,
        "old_position_Einf_sq":
            old_position_error(
                POSITION_REP_SOURCE[label],
                RHO_X_EXACT_SOURCE
            )
    })

OLD_AUDIT = pd.DataFrame(old_rows)
display(OLD_AUDIT)

EXPECTED_OLD = {
    "KQSE_OLD": 0.03287541656187098,
    "BI10": 0.07408970300536183,
    "CO10": 0.05655951446607987,
    "NN10": 0.061073038435645294,
    "BI20": 0.0044776820539136515,
    "CO20": 0.007756449095745787,
    "NN20": 0.001331183729228106,
}

audit_diff_rows = []

for _, row in OLD_AUDIT.iterrows():
    lab = row["result"]
    expected = EXPECTED_OLD[lab]
    actual = float(row["old_position_Einf_sq"])

    audit_diff_rows.append({
        "result": lab,
        "actual": actual,
        "expected_from_executed_coauthor_plot_notebook": expected,
        "abs_difference": abs(actual-expected),
    })

OLD_AUDIT_CHECK = pd.DataFrame(audit_diff_rows)
display(OLD_AUDIT_CHECK)

assert OLD_AUDIT_CHECK["abs_difference"].max() < 1e-10

print("HISTORICAL POSITION-KERNEL AUDIT PASSED.")

OLD_AUDIT.to_csv(
    OUTDIR/"old_position_metric_audit_corrected.csv",
    index=False
)

## 4. Convert the source-array position kernels back to a common Fock basis

A subtle array-orientation issue matters here. In the coauthors' exact plotting code,

```python
Y, YP = np.meshgrid(y_array, y_array)
```

uses NumPy's default `indexing="xy"`. Hence the displayed theoretical array has
rows labelled by \(y'\) and columns labelled by \(y\); it is the transpose of the
standard matrix representation \(K_{ij}=\rho(y_i,y_j)\).

The old plotting transformations are first reproduced exactly in that **source-array
orientation**. Before recovering a Fock density matrix, each represented kernel is therefore
transposed back to the standard kernel orientation.

This resolves the apparent contradiction seen in the first Stage-2 run and reproduces the
coauthors' historical \(N=10\) and \(N=20\) pointwise errors exactly.

In [ ]:
def recover_fock_from_position_kernel(K_standard, x_grid, dim):
    """
    Recover a Fock density matrix from a position kernel in STANDARD
    matrix orientation K[i,j] = rho(x_i,x_j).
    """
    K = np.asarray(K_standard, complex)
    Psi = ho_basis_matrix(x_grid, dim)
    A = Psi.T
    Aplus = np.linalg.pinv(A)

    rho = Aplus @ K @ Aplus.T
    K_back = A @ rho @ A.T

    residual = (
        np.linalg.norm(K_back-K)
        /np.linalg.norm(K)
    )

    return rho, residual

DIMS = {
    "BI10":11,
    "BI20":21,
    "CO10":10,
    "CO20":20,
    "NN10":10,
    "NN20":20,
}

RHO_CANON={}
recovery_rows=[]

for label,dim in DIMS.items():

    # IMPORTANT:
    # POSITION_REP_SOURCE is in the coauthors' displayed/source-array
    # orientation [row=y', column=y].  Transpose it back to the
    # standard kernel orientation before Fock recovery.
    K_standard = POSITION_REP_SOURCE[label].T

    rho,res = recover_fock_from_position_kernel(
        K_standard,
        X_POS,
        dim
    )

    RHO_CANON[label]=rho

    # Independent round-trip check in source-array orientation.
    K_roundtrip_standard = position_kernel_from_fock(
        rho,
        X_POS
    )
    K_roundtrip_source = K_roundtrip_standard.T

    source_array_residual = (
        np.linalg.norm(
            K_roundtrip_source
            - POSITION_REP_SOURCE[label]
        )
        / np.linalg.norm(
            POSITION_REP_SOURCE[label]
        )
    )

    rho_h=0.5*(rho+rho.conj().T)
    eig=np.linalg.eigvalsh(rho_h)

    recovery_rows.append({
        "result":label,
        "dim":dim,
        "kernel_recovery_residual":res,
        "source_array_roundtrip_residual":
            source_array_residual,
        "trace_real":
            float(np.real(np.trace(rho))),
        "hermiticity_max":
            float(np.max(
                np.abs(rho-rho.conj().T)
            )),
        "min_eigenvalue_hermitized":
            float(np.min(eig)),
        "purity_hermitized":
            float(np.real(
                np.trace(rho_h@rho_h)
            )),
    })

RECOVERY_TABLE=pd.DataFrame(recovery_rows)
display(RECOVERY_TABLE)

RECOVERY_TABLE.to_csv(
    OUTDIR/"represented_fock_recovery_diagnostics_corrected.csv",
    index=False
)

assert np.max(
    RECOVERY_TABLE["kernel_recovery_residual"]
) < 1e-8

assert np.max(
    RECOVERY_TABLE["source_array_roundtrip_residual"]
) < 1e-8

print("SOURCE-ARRAY -> STANDARD FOCK RECOVERY PASSED.")

## 5. Independently validated Fock \(\rho\to W,w_r\) converter

The self-test includes the complex cat state, so a hidden
\(p\to-p\) or complex-conjugation error cannot pass unnoticed.

In [ ]:
def wigner_from_fock_rho_fast(rho, q_grid, p_grid):
    rho = np.asarray(rho, complex)
    dim = rho.shape[0]

    Q,P=np.meshgrid(
        np.asarray(q_grid,float),
        np.asarray(p_grid,float),
        indexing="ij"
    )

    B=np.sqrt(2.0)*(Q+1j*P)
    R2=np.abs(B)**2
    G=np.exp(-R2/2.0)

    total=np.zeros_like(B,dtype=complex)

    for i in range(dim):
        parity_i=(-1.0)**i

        for j in range(dim):
            if j>=i:
                fr=np.exp(
                    0.5*(
                        gammaln(i+1)
                        -gammaln(j+1)
                    )
                )
                Dji=(
                    G*fr
                    *B**(j-i)
                    *eval_genlaguerre(
                        i,j-i,R2
                    )
                )
            else:
                fr=np.exp(
                    0.5*(
                        gammaln(j+1)
                        -gammaln(i+1)
                    )
                )
                Dji=(
                    G*fr
                    *(-np.conj(B))**(i-j)
                    *eval_genlaguerre(
                        j,i-j,R2
                    )
                )

            total += (
                rho[i,j]
                *parity_i
                *Dji
            )

    return np.real(total)/np.pi

def displacement_element(n,m,beta):
    beta=complex(beta)
    r2=abs(beta)**2
    G=np.exp(-r2/2.0)

    if n>=m:
        fr=np.exp(
            0.5*(
                gammaln(m+1)
                -gammaln(n+1)
            )
        )
        return (
            G*fr
            *beta**(n-m)
            *eval_genlaguerre(
                m,n-m,r2
            )
        )

    fr=np.exp(
        0.5*(
            gammaln(n+1)
            -gammaln(m+1)
        )
    )
    return (
        G*fr
        *(-np.conj(beta))**(m-n)
        *eval_genlaguerre(
            n,m-n,r2
        )
    )

def photon_from_fock_rho(rho,alpha_grid,r):
    rho=np.asarray(rho,complex)
    dim=rho.shape[0]

    out=np.empty(
        len(alpha_grid),float
    )

    for ia,alpha in enumerate(
        np.asarray(alpha_grid,complex)
    ):
        v=np.array([
            displacement_element(
                r,m,alpha
            )
            for m in range(dim)
        ],dtype=complex)

        out[ia]=np.real(
            v@rho@np.conj(v)
        )

    return out

In [ ]:
def exact_cat_fock_rho(dim=60,a=A_CAT):
    n=np.arange(dim)
    sqrt_fact=np.exp(
        0.5*gammaln(n+1)
    )

    psi=np.zeros(dim,complex)

    for amp in cat_amplitudes(a):
        psi += (
            np.exp(-abs(amp)**2/2)
            *amp**n
            /sqrt_fact
        )

    psi /= np.linalg.norm(psi)

    return np.outer(
        psi,np.conj(psi)
    )

rho_exact_fock=exact_cat_fock_rho(
    dim=60
)

q_test=np.linspace(-5.5,5.5,31)
p_test=np.linspace(-5.5,5.5,31)

W_test_num=wigner_from_fock_rho_fast(
    rho_exact_fock,
    q_test,
    p_test
)

W_test_exact=np.real(
    exact_cat_wigner(
        q_test,
        p_test
    )
)

err_W_complex_cat=float(
    np.max(
        np.abs(
            W_test_num-W_test_exact
        )
    )
)

alpha_test=np.linspace(
    -5.5,5.5,41
).astype(complex)

ph_errors={}

for r in R_LIST:
    w_num=photon_from_fock_rho(
        rho_exact_fock,
        alpha_test,
        r
    )
    w_true=exact_cat_photon(
        alpha_test,
        r
    )
    ph_errors[r]=float(
        np.max(
            np.abs(w_num-w_true)
        )
    )

print(
    "Complex-cat Wigner max abs error:",
    err_W_complex_cat
)
print(
    "Complex-cat photon max abs errors:",
    ph_errors
)

assert err_W_complex_cat < 1e-8
assert max(ph_errors.values()) < 1e-8

print(
    "COMPLEX-CAT CONVERTER SELF-TEST PASSED."
)

## 6. Current frozen KQSE on the same `cat_state_data.dat`

The original 72 phases cover \(0\le\theta<2\pi\). Antipodal data are folded into
36 directions using \(x_{\theta+\pi}=-x_\theta\), giving 4000 measurements per
folded phase bin and retaining all \(144000\) measurements.

In [ ]:
N_THETA_FOLDED=36
BIN_WIDTH=np.pi/N_THETA_FOLDED

theta_mod=np.mod(
    theta_data,np.pi
)

half_index=(
    np.floor(
        theta_data/np.pi
    ).astype(int)%2
)

fold_sign=np.where(
    half_index==0,
    1.0,
    -1.0
)

bin_idx=np.floor(
    theta_mod/BIN_WIDTH
).astype(int)

bin_idx=np.clip(
    bin_idx,
    0,
    N_THETA_FOLDED-1
)

GROUPS={
    b: (
        x_data[bin_idx==b]
        *fold_sign[bin_idx==b]
    )
    for b in range(
        N_THETA_FOLDED
    )
}

counts=np.array([
    len(GROUPS[b])
    for b in range(
        N_THETA_FOLDED
    )
])

print("Folded counts:",counts)
print(
    "min/max:",
    counts.min(),
    counts.max()
)
print(
    "total:",
    counts.sum()
)

assert counts.min()==4000
assert counts.max()==4000
assert counts.sum()==144000

In [ ]:
def normalize_pdf_on_grid(pdf,xgrid):
    pdf=np.maximum(
        np.asarray(pdf,float),
        0.0
    )
    Z=np.trapezoid(
        pdf,xgrid
    )
    if (
        not np.isfinite(Z)
        or Z<=0
    ):
        raise ValueError(
            "Bad PDF normalization"
        )
    return pdf/Z

margin=1.0

X_PRECISION=np.linspace(
    float(np.min(x_data)-margin),
    float(np.max(x_data)+margin),
    N_PRECISION
)

PILOT_PDF={}
PILOT_MASS={}

for b in tqdm(
    range(N_THETA_FOLDED),
    desc="Pilot KDEs"
):
    samples=np.asarray(
        GROUPS[b],float
    )

    kde=gaussian_kde(
        samples,
        bw_method="silverman"
    )

    raw_pdf=kde.evaluate(
        X_PRECISION
    )

    PILOT_MASS[b]=float(
        np.trapezoid(
            raw_pdf,
            X_PRECISION
        )
    )

    PILOT_PDF[b]=\
        normalize_pdf_on_grid(
            raw_pdf,
            X_PRECISION
        )

print(
    "Pilot mass before renormalization min/max:",
    min(PILOT_MASS.values()),
    max(PILOT_MASS.values())
)

In [ ]:
def centered_endpoint_excluded_grid(R,N):
    delta=2.0*float(R)/int(N)
    return (
        np.arange(int(N))
        -(int(N)-1)/2.0
    )*delta

def pilot_phi_at_t(pdf,t):
    return np.trapezoid(
        pdf*np.exp(
            1j*X_PRECISION*t
        ),
        X_PRECISION
    )

def canonical_theta_radius_sign(mu,nu):
    radius=float(
        np.hypot(mu,nu)
    )

    if radius<1e-12:
        return 0.0,0.0,1.0

    theta=float(
        np.arctan2(nu,mu)
    )

    if theta<0:
        theta += 2*np.pi

    if theta>=np.pi:
        return (
            theta-np.pi,
            radius,
            -1.0
        )

    return theta,radius,+1.0

def kqse_phi_point(mu,nu):
    theta,radius,sample_sign=\
        canonical_theta_radius_sign(
            mu,nu
        )

    if radius<1e-12:
        return 1.0+0.0j,0.0

    b=int(
        np.floor(
            theta/BIN_WIDTH
        )
    )
    b=min(
        max(b,0),
        N_THETA_FOLDED-1
    )

    base_samples=np.asarray(
        GROUPS[b],float
    )

    phi_pilot=pilot_phi_at_t(
        PILOT_PDF[b],
        radius
    )

    abs_phi=max(
        float(np.abs(phi_pilot)),
        PHI_ABS_FLOOR
    )

    numer=max(
        0.0,
        1.0-abs_phi**2
    )

    denom=(
        len(base_samples)
        *abs_phi**2
    )

    raw_h=(
        2.0/abs(radius)
        *np.sqrt(
            numer/denom
        )
    )

    h=float(
        np.clip(
            raw_h,
            H_MIN,
            H_MAX
        )
    )

    xb=(
        sample_sign
        *base_samples
    )

    phi_emp=np.mean(
        np.exp(
            1j*xb*radius
        )
    )

    phi_K=np.exp(
        -h**2*radius**2/4.0
    )

    return phi_emp*phi_K,h

MU_GRID=centered_endpoint_excluded_grid(
    PHI_R,PHI_N
)
NU_GRID=centered_endpoint_excluded_grid(
    PHI_R,PHI_N
)

PHI_KQSE=np.zeros(
    (PHI_N,PHI_N),
    complex
)
H_GRID=np.zeros(
    (PHI_N,PHI_N),
    float
)

for i,mu in enumerate(
    tqdm(
        MU_GRID,
        desc="Current KQSE CF"
    )
):
    for j,nu in enumerate(
        NU_GRID
    ):
        PHI_KQSE[i,j],H_GRID[i,j]=\
            kqse_phi_point(
                mu,nu
            )

PHI_KQSE=0.5*(
    PHI_KQSE
    +np.conj(
        PHI_KQSE[::-1,::-1]
    )
)

i0=np.argmin(
    np.abs(MU_GRID)
)
j0=np.argmin(
    np.abs(NU_GRID)
)
PHI_KQSE[i0,j0]=1.0+0.0j

positive_h=H_GRID[
    H_GRID>0
]

print(
    "Hermitian defect:",
    np.max(
        np.abs(
            PHI_KQSE
            -np.conj(
                PHI_KQSE[::-1,::-1]
            )
        )
    )
)
print(
    "h min/median/max:",
    np.min(positive_h),
    np.median(positive_h),
    np.max(positive_h)
)
print(
    "fraction at H_MAX:",
    np.mean(
        np.isclose(
            positive_h,H_MAX
        )
    )
)

In [ ]:
def wigner_from_phi_grid(
    phi_grid,
    mu_grid,
    nu_grid,
    q_grid,
    p_grid
):
    dmu=float(
        mu_grid[1]-mu_grid[0]
    )
    dnu=float(
        nu_grid[1]-nu_grid[0]
    )

    MU,NU=np.meshgrid(
        mu_grid,
        nu_grid,
        indexing="ij"
    )

    W=np.empty(
        (
            len(q_grid),
            len(p_grid)
        ),
        complex
    )

    for iq,q in enumerate(
        tqdm(
            q_grid,
            desc="KQSE Wigner"
        )
    ):
        for ip,p in enumerate(
            p_grid
        ):
            phase=np.exp(
                -1j*(
                    MU*q+NU*p
                )
            )

            W[iq,ip]=(
                dmu*dnu
                /(2*np.pi)**2
            )*np.sum(
                phi_grid*phase
            )

    return np.real_if_close(W)

def photon_kernel(
    alpha,mu,nu,r
):
    radial=np.exp(
        -(mu**2+nu**2)/4.0
    )

    phase=np.exp(
        -(
            (nu-1j*mu)
            /np.sqrt(2.0)
        )*np.conj(alpha)
        +(
            (nu+1j*mu)
            /np.sqrt(2.0)
        )*alpha
    )

    lag=eval_genlaguerre(
        r,0,
        (mu**2+nu**2)/2.0
    )

    return radial*phase*lag

def photon_from_phi_grid(
    phi_grid,
    mu_grid,
    nu_grid,
    alpha_grid,
    r
):
    dmu=float(
        mu_grid[1]-mu_grid[0]
    )
    dnu=float(
        nu_grid[1]-nu_grid[0]
    )

    MU,NU=np.meshgrid(
        mu_grid,
        nu_grid,
        indexing="ij"
    )

    out=np.empty(
        len(alpha_grid),
        complex
    )

    for ia,alpha in enumerate(
        tqdm(
            alpha_grid,
            desc=f"KQSE photon r={r}"
        )
    ):
        K=photon_kernel(
            alpha,MU,NU,r
        )

        out[ia]=(
            dmu*dnu/(2*np.pi)
        )*np.sum(
            phi_grid*K
        )

    return np.real_if_close(out)

W_KQSE=np.real(
    wigner_from_phi_grid(
        PHI_KQSE,
        MU_GRID,
        NU_GRID,
        Q_GRID,
        P_GRID
    )
)

w_KQSE={
    r:np.real(
        photon_from_phi_grid(
            PHI_KQSE,
            MU_GRID,
            NU_GRID,
            ALPHA_GRID,
            r
        )
    )
    for r in R_LIST
}

print(
    "Current KQSE observables built."
)

## 7. Convert BI / CO / NN(CGAN) to Wigner and photon-number tomograms

In [ ]:
W_EXT={}
w_EXT={}

for label,rho in RHO_CANON.items():
    print(
        "Converting",label
    )

    W_EXT[label]=\
        wigner_from_fock_rho_fast(
            rho,
            Q_GRID,
            P_GRID
        )

    w_EXT[label]={
        r:photon_from_fock_rho(
            rho,
            ALPHA_GRID,
            r
        )
        for r in R_LIST
    }

print(
    "External-method observables built."
)

## 8. Common single-reconstruction metrics

For this fixed common synthetic dataset we report ISE and

\[
E_\infty=
\max_z
|\widehat\Theta(z)-\Theta_{\rm exact}(z)|^2.
\]

In [ ]:
def ise_2d(A,B,x,y):
    dx=float(x[1]-x[0])
    dy=float(y[1]-y[0])

    return float(
        dx*dy*np.sum(
            np.abs(
                np.asarray(A)
                -np.asarray(B)
            )**2
        )
    )

def ise_1d(A,B,x):
    dx=float(
        np.real(
            x[1]-x[0]
        )
    )

    return float(
        dx*np.sum(
            np.abs(
                np.asarray(A)
                -np.asarray(B)
            )**2
        )
    )

def e_inf_sq(A,B):
    return float(
        np.max(
            np.abs(
                np.asarray(A)
                -np.asarray(B)
            )**2
        )
    )

METHOD_LABELS={
    "BI10":"BI",
    "CO10":"CO",
    "NN10":"CGAN",
    "BI20":"BI",
    "CO20":"CO",
    "NN20":"CGAN",
}

rows=[]

for cutoff,labels in [
    (10,["BI10","CO10","NN10"]),
    (20,["BI20","CO20","NN20"]),
]:
    rows.append({
        "Cutoff":cutoff,
        "Observable":"Wigner",
        "Method":"KQSE",
        "ISE":ise_2d(
            W_KQSE,W_EXACT,
            Q_GRID,P_GRID
        ),
        "E_inf":e_inf_sq(
            W_KQSE,W_EXACT
        ),
    })

    for r in R_LIST:
        rows.append({
            "Cutoff":cutoff,
            "Observable":f"w_{r}",
            "Method":"KQSE",
            "ISE":ise_1d(
                w_KQSE[r],
                w_EXACT[r],
                ALPHA_GRID
            ),
            "E_inf":e_inf_sq(
                w_KQSE[r],
                w_EXACT[r]
            ),
        })

    for label in labels:
        method=METHOD_LABELS[label]

        rows.append({
            "Cutoff":cutoff,
            "Observable":"Wigner",
            "Method":method,
            "ISE":ise_2d(
                W_EXT[label],
                W_EXACT,
                Q_GRID,P_GRID
            ),
            "E_inf":e_inf_sq(
                W_EXT[label],
                W_EXACT
            ),
        })

        for r in R_LIST:
            rows.append({
                "Cutoff":cutoff,
                "Observable":f"w_{r}",
                "Method":method,
                "ISE":ise_1d(
                    w_EXT[label][r],
                    w_EXACT[r],
                    ALPHA_GRID
                ),
                "E_inf":e_inf_sq(
                    w_EXT[label][r],
                    w_EXACT[r]
                ),
            })

FINAL_TABLE=pd.DataFrame(rows)

display(FINAL_TABLE)

FINAL_TABLE.to_csv(
    OUTDIR/
    "synthetic_coauthor_KQSE_BI_CO_CGAN_N10_N20.csv",
    index=False
)

for cutoff in [10,20]:
    print(
        "\n=============================="
    )
    print(
        "Cutoff",cutoff
    )
    print(
        "=============================="
    )

    sub=FINAL_TABLE[
        FINAL_TABLE["Cutoff"]==cutoff
    ]

    for obs in [
        "Wigner",
        "w_0","w_1","w_2","w_3"
    ]:
        s=sub[
            sub["Observable"]==obs
        ]

        b1=s.loc[
            s["ISE"].idxmin()
        ]
        b2=s.loc[
            s["E_inf"].idxmin()
        ]

        print(
            f"{obs:7s} | "
            f"best ISE: {b1['Method']} ({b1['ISE']:.3e}) | "
            f"best E_inf: {b2['Method']} ({b2['E_inf']:.3e})"
        )

## 9. Preliminary Wigner figures

In [ ]:
for cutoff,labels in [
    (10,["BI10","CO10","NN10"]),
    (20,["BI20","CO20","NN20"]),
]:
    panels=[
        ("(a) Exact",W_EXACT),
        ("(b) KQSE",W_KQSE),
        ("(c) BI",W_EXT[labels[0]]),
        ("(d) CO",W_EXT[labels[1]]),
        ("(e) CGAN",W_EXT[labels[2]]),
    ]

    vmin=min(
        float(np.min(W))
        for _,W in panels
    )
    vmax=max(
        float(np.max(W))
        for _,W in panels
    )

    fig,axes=plt.subplots(
        2,3,
        figsize=(16.5,10.5),
        constrained_layout=True
    )
    axes=axes.ravel()

    for ax,(title,W) in zip(
        axes,panels
    ):
        im=ax.imshow(
            np.asarray(W).T,
            origin="lower",
            extent=[
                Q_GRID[0],Q_GRID[-1],
                P_GRID[0],P_GRID[-1]
            ],
            aspect="equal",
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(
            title,
            fontsize=20
        )
        ax.set_xlabel(
            r"$q$",
            fontsize=18
        )
        ax.set_ylabel(
            r"$p$",
            fontsize=18
        )
        ax.tick_params(
            labelsize=13
        )

    axes[-1].axis("off")

    cbar=fig.colorbar(
        im,
        ax=axes[:-1].tolist(),
        shrink=.92
    )
    cbar.set_label(
        r"$W(q,p)$",
        fontsize=18
    )
    cbar.ax.tick_params(
        labelsize=13
    )

    path=OUTDIR/f"fig_synthetic_coauthor_wigner_N{cutoff}.pdf"

    fig.savefig(
        path,
        bbox_inches="tight"
    )
    plt.show()

    print(
        "Saved:",path
    )

## 10. Preliminary photon-number figures

In [ ]:
for cutoff,labels in [
    (10,["BI10","CO10","NN10"]),
    (20,["BI20","CO20","NN20"]),
]:
    fig,axes=plt.subplots(
        2,2,
        figsize=(13.8,10.0),
        constrained_layout=True
    )
    axes=axes.ravel()

    for ax,r in zip(
        axes,R_LIST
    ):
        ax.plot(
            ALPHA_GRID.real,
            w_EXACT[r],
            linewidth=2.7,
            label="Exact"
        )

        ax.plot(
            ALPHA_GRID.real,
            w_KQSE[r],
            linewidth=2.3,
            linestyle="--",
            label="KQSE"
        )

        ax.plot(
            ALPHA_GRID.real,
            w_EXT[labels[0]][r],
            linewidth=2.0,
            linestyle=":",
            label="BI"
        )

        ax.plot(
            ALPHA_GRID.real,
            w_EXT[labels[1]][r],
            linewidth=2.0,
            linestyle="-.",
            label="CO"
        )

        ax.plot(
            ALPHA_GRID.real,
            w_EXT[labels[2]][r],
            linewidth=2.0,
            linestyle=(0,(3,1,1,1)),
            label="CGAN"
        )

        ax.set_title(
            rf"$r={r}$",
            fontsize=19
        )
        ax.set_xlabel(
            r"$\mathrm{Re}\,\alpha$",
            fontsize=17
        )
        ax.set_ylabel(
            rf"$\mathsf{{w}}_{{{r}}}(\alpha)$",
            fontsize=17
        )
        ax.tick_params(
            labelsize=12
        )
        ax.grid(
            alpha=.25
        )

    handles,leglabels=\
        axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        leglabels,
        loc="upper center",
        ncol=5,
        fontsize=13,
        frameon=True,
        bbox_to_anchor=(.5,1.035)
    )

    path=OUTDIR/f"fig_synthetic_coauthor_photon_N{cutoff}.pdf"

    fig.savefig(
        path,
        bbox_inches="tight"
    )
    plt.show()

    print(
        "Saved:",path
    )

## 11. Article-friendly pivot tables

In [ ]:
for cutoff in [10,20]:
    sub=FINAL_TABLE[
        FINAL_TABLE["Cutoff"]==cutoff
    ].copy()

    print(
        "\nISE, cutoff =",cutoff
    )
    display(
        sub.pivot(
            index="Observable",
            columns="Method",
            values="ISE"
        )
    )

    print(
        "\nE_inf, cutoff =",cutoff
    )
    display(
        sub.pivot(
            index="Observable",
            columns="Method",
            values="E_inf"
        )
    )

## What to send back

Please send the executed notebook, or at minimum:

1. `OLD_AUDIT`;
2. `RECOVERY_TABLE`;
3. complex-cat converter self-test;
4. folded counts and bandwidth diagnostics for the current KQSE;
5. `FINAL_TABLE`;
6. the winner summaries;
7. the four generated PDFs if convenient.

**First inspect the historical audit.** If the source-derived position kernels do not reproduce
the old \(N=10\) and \(N=20\) errors reasonably well, stop and do not use the new observable
comparison yet.

# Stage 3 pilot — old KQSE vs current KQSE, plus 10-run stability check

Purpose: decide whether a new \(R=100\) run is actually useful for the **coauthor benchmark**.

This block does **not** rerun BI / CO / CGAN.

It performs two diagnostics:

1. On the original common `cat_state_data.dat`, convert the historical saved `KQSE.npy`
   position kernel directly to a Wigner function and compare it with the **current** KQSE
   Wigner reconstruction on the same data.
2. Run a paired **within-phase bootstrap** of the current frozen KQSE estimator
   (`approx h`, floor 0.03, clip 0.01--0.5) for 10 resamples of the same 144000-point
   synthetic dataset. This estimates how atypical the single current-KQSE reconstruction is.

The bootstrap is a **stability diagnostic**, not a replacement for a genuine Monte Carlo
MISE/MSE study. In particular, BI/CO/CGAN remain single saved reconstructions.

In [ ]:
# ================================================================
# 12. Position-kernel -> Wigner converter, independently validated
# ================================================================

from scipy.interpolate import RegularGridInterpolator

def wigner_from_position_kernel(
    K_standard,
    x_grid,
    q_grid,
    p_grid,
    s_points=801,
    phase_sign=-1,
):
    # Standard kernel convention: K_standard[i,j] = rho(x_i, x_j).
    K_standard = np.asarray(K_standard, complex)
    x_grid = np.asarray(x_grid, float)
    q_grid = np.asarray(q_grid, float)
    p_grid = np.asarray(p_grid, float)

    interp_re = RegularGridInterpolator(
        (x_grid, x_grid),
        K_standard.real,
        bounds_error=False,
        fill_value=0.0,
    )
    interp_im = RegularGridInterpolator(
        (x_grid, x_grid),
        K_standard.imag,
        bounds_error=False,
        fill_value=0.0,
    )

    s_grid = np.linspace(
        -2.0*(x_grid[-1]-x_grid[0])/2.0,
        +2.0*(x_grid[-1]-x_grid[0])/2.0,
        int(s_points)
    )

    W = np.empty((len(q_grid), len(p_grid)), complex)

    phase_matrix = np.exp(
        phase_sign * 1j
        * np.outer(p_grid, s_grid)
    )

    for iq, q in enumerate(tqdm(q_grid, desc="position-kernel -> Wigner")):
        x1 = q + s_grid/2.0
        x2 = q - s_grid/2.0
        pts = np.column_stack([x1, x2])

        line = (
            interp_re(pts)
            + 1j*interp_im(pts)
        )

        W[iq, :] = np.trapezoid(
            phase_matrix * line[None, :],
            s_grid,
            axis=1,
        ) / (2*np.pi)

    return W


# Validate the Fourier sign/orientation against the exact complex cat state.
W_pos_minus = wigner_from_position_kernel(
    RHO_X_EXACT_STANDARD,
    X_POS,
    Q_GRID,
    P_GRID,
    s_points=801,
    phase_sign=-1,
)
W_pos_plus = wigner_from_position_kernel(
    RHO_X_EXACT_STANDARD,
    X_POS,
    Q_GRID,
    P_GRID,
    s_points=801,
    phase_sign=+1,
)

err_minus = float(np.max(np.abs(np.real(W_pos_minus)-W_EXACT)))
err_plus = float(np.max(np.abs(np.real(W_pos_plus)-W_EXACT)))

print("Exact-kernel validation:")
print("phase_sign=-1 max abs error =", err_minus)
print("phase_sign=+1 max abs error =", err_plus)

POSITION_WIGNER_PHASE_SIGN = -1 if err_minus <= err_plus else +1
POSITION_WIGNER_VALIDATION_ERROR = min(err_minus, err_plus)

print("Chosen phase sign:", POSITION_WIGNER_PHASE_SIGN)
print("Validation error:", POSITION_WIGNER_VALIDATION_ERROR)

assert POSITION_WIGNER_VALIDATION_ERROR < 2e-3

In [ ]:
# ================================================================
# 13. Historical KQSE Wigner vs current KQSE Wigner on SAME data
# ================================================================

KQSE_OLD_STANDARD = POSITION_REP_SOURCE["KQSE_OLD"].T

W_KQSE_OLD_COMPLEX = wigner_from_position_kernel(
    KQSE_OLD_STANDARD,
    X_POS,
    Q_GRID,
    P_GRID,
    s_points=801,
    phase_sign=POSITION_WIGNER_PHASE_SIGN,
)

W_KQSE_OLD = np.real(W_KQSE_OLD_COMPLEX)

old_imag_leakage = float(
    np.max(np.abs(np.imag(W_KQSE_OLD_COMPLEX)))
)

OLD_VS_CURRENT_WIGNER = pd.DataFrame([
    {
        "KQSE_version": "Historical coauthor KQSE",
        "Wigner_ISE": ise_2d(
            W_KQSE_OLD, W_EXACT, Q_GRID, P_GRID
        ),
        "Wigner_E_inf_sq": e_inf_sq(
            W_KQSE_OLD, W_EXACT
        ),
        "max_abs_imaginary_Wigner": old_imag_leakage,
    },
    {
        "KQSE_version": "Current frozen KQSE",
        "Wigner_ISE": ise_2d(
            W_KQSE, W_EXACT, Q_GRID, P_GRID
        ),
        "Wigner_E_inf_sq": e_inf_sq(
            W_KQSE, W_EXACT
        ),
        "max_abs_imaginary_Wigner": 0.0,
    },
])

display(OLD_VS_CURRENT_WIGNER)

print(
    "\nHistorical density-kernel E_inf^2 on the original coauthor metric =",
    float(
        OLD_AUDIT.loc[
            OLD_AUDIT["result"]=="KQSE_OLD",
            "old_position_Einf_sq"
        ].iloc[0]
    )
)

OLD_VS_CURRENT_WIGNER.to_csv(
    OUTDIR/"KQSE_old_vs_current_same_data_Wigner.csv",
    index=False
)

## 14. Bootstrap design

The 36 folded phase bins each contain 4000 measurements. For every bootstrap replicate we
sample **4000 points with replacement inside each folded phase bin**, then rerun the entire
current KQSE pipeline including the Gaussian-KDE pilot and adaptive bandwidth.

Default:
\[
B=10.
\]

This is intentionally much cheaper than \(R=100\), while already giving a useful estimate
of the variability of the current KQSE result.

In [ ]:
# ================================================================
# 14A. Fast observable transforms for repeated runs
# ================================================================

N_BOOT = 10
BOOT_SEEDS = list(range(100, 100 + N_BOOT))

E_MU_Q = np.exp(
    -1j * np.outer(MU_GRID, Q_GRID)
)
E_NU_P = np.exp(
    -1j * np.outer(NU_GRID, P_GRID)
)
CF_TO_W_FACTOR = (
    float(MU_GRID[1]-MU_GRID[0])
    * float(NU_GRID[1]-NU_GRID[0])
    / (2*np.pi)**2
)

MU_MESH, NU_MESH = np.meshgrid(
    MU_GRID, NU_GRID, indexing="ij"
)
PHOTON_KERNEL_CACHE = {}

for r in R_LIST:
    K_stack = np.empty(
        (len(ALPHA_GRID), PHI_N, PHI_N),
        complex
    )
    for ia, alpha in enumerate(ALPHA_GRID):
        K_stack[ia] = photon_kernel(
            alpha, MU_MESH, NU_MESH, r
        )
    PHOTON_KERNEL_CACHE[r] = K_stack

PHOTON_FACTOR = (
    float(MU_GRID[1]-MU_GRID[0])
    * float(NU_GRID[1]-NU_GRID[0])
    / (2*np.pi)
)

def observables_fast_from_phi(phi):
    W = CF_TO_W_FACTOR * (
        E_MU_Q.T @ phi @ E_NU_P
    )
    W = np.real(W)

    w = {}
    for r in R_LIST:
        w[r] = np.real(
            PHOTON_FACTOR
            * np.einsum(
                "aij,ij->a",
                PHOTON_KERNEL_CACHE[r],
                phi,
                optimize=True,
            )
        )

    return W, w


W_fast_check, w_fast_check = observables_fast_from_phi(PHI_KQSE)

print(
    "Fast Wigner transform max difference:",
    np.max(np.abs(W_fast_check-W_KQSE))
)
for r in R_LIST:
    print(
        f"Fast photon r={r} max difference:",
        np.max(np.abs(w_fast_check[r]-w_KQSE[r]))
    )

assert np.max(np.abs(W_fast_check-W_KQSE)) < 1e-10
assert max(
    np.max(np.abs(w_fast_check[r]-w_KQSE[r]))
    for r in R_LIST
) < 1e-10

In [ ]:
# ================================================================
# 14B. One complete current-KQSE bootstrap reconstruction
# ================================================================

def build_current_kqse_from_groups(groups_local):
    pilot_pdf_local = {}

    for b in range(N_THETA_FOLDED):
        samples = np.asarray(groups_local[b], float)

        kde = gaussian_kde(
            samples,
            bw_method="silverman"
        )

        raw_pdf = kde.evaluate(
            X_PRECISION
        )

        pilot_pdf_local[b] = normalize_pdf_on_grid(
            raw_pdf,
            X_PRECISION
        )

    phi = np.zeros(
        (PHI_N, PHI_N),
        complex
    )
    hgrid = np.zeros(
        (PHI_N, PHI_N),
        float
    )

    for i, mu in enumerate(MU_GRID):
        for j, nu in enumerate(NU_GRID):
            theta, radius, sample_sign = \
                canonical_theta_radius_sign(mu, nu)

            if radius < 1e-12:
                phi[i,j] = 1.0 + 0.0j
                hgrid[i,j] = 0.0
                continue

            b = int(
                np.floor(theta/BIN_WIDTH)
            )
            b = min(
                max(b, 0),
                N_THETA_FOLDED-1
            )

            base_samples = np.asarray(
                groups_local[b], float
            )

            phi_pilot = np.trapezoid(
                pilot_pdf_local[b]
                * np.exp(
                    1j*X_PRECISION*radius
                ),
                X_PRECISION
            )

            abs_phi = max(
                float(np.abs(phi_pilot)),
                PHI_ABS_FLOOR
            )

            numer = max(
                0.0,
                1.0-abs_phi**2
            )
            denom = (
                len(base_samples)
                * abs_phi**2
            )

            raw_h = (
                2.0/abs(radius)
                * np.sqrt(numer/denom)
            )

            h = float(
                np.clip(
                    raw_h,
                    H_MIN,
                    H_MAX
                )
            )

            xb = (
                sample_sign
                * base_samples
            )

            phi_emp = np.mean(
                np.exp(
                    1j*xb*radius
                )
            )

            phi_K = np.exp(
                -h**2*radius**2/4.0
            )

            phi[i,j] = (
                phi_emp*phi_K
            )
            hgrid[i,j] = h

    phi = 0.5*(
        phi
        + np.conj(
            phi[::-1, ::-1]
        )
    )

    phi[i0,j0] = 1.0+0.0j

    W, w = observables_fast_from_phi(phi)

    return phi, hgrid, W, w

In [ ]:
# ================================================================
# 14C. Run B=10 bootstrap replicates
# ================================================================

boot_rows = []

for rep, seed in enumerate(BOOT_SEEDS):
    print(f"\nBootstrap {rep+1}/{N_BOOT}, seed={seed}")

    rng = np.random.default_rng(seed)

    groups_boot = {}

    for b in range(N_THETA_FOLDED):
        base = np.asarray(GROUPS[b], float)
        idx = rng.integers(
            0, len(base), size=len(base)
        )
        groups_boot[b] = base[idx]

    phi_b, h_b, W_b, w_b = \
        build_current_kqse_from_groups(
            groups_boot
        )

    positive_h = h_b[h_b>0]

    boot_rows.append({
        "rep": rep,
        "seed": seed,
        "Observable": "Wigner",
        "ISE": ise_2d(
            W_b, W_EXACT, Q_GRID, P_GRID
        ),
        "E_inf": e_inf_sq(
            W_b, W_EXACT
        ),
        "h_median": float(
            np.median(positive_h)
        ),
        "frac_at_H_MAX": float(
            np.mean(
                np.isclose(
                    positive_h,
                    H_MAX
                )
            )
        ),
    })

    for r in R_LIST:
        boot_rows.append({
            "rep": rep,
            "seed": seed,
            "Observable": f"w_{r}",
            "ISE": ise_1d(
                w_b[r],
                w_EXACT[r],
                ALPHA_GRID
            ),
            "E_inf": e_inf_sq(
                w_b[r],
                w_EXACT[r]
            ),
            "h_median": float(
                np.median(positive_h)
            ),
            "frac_at_H_MAX": float(
                np.mean(
                    np.isclose(
                        positive_h,
                        H_MAX
                    )
                )
            ),
        })

BOOT_TABLE = pd.DataFrame(boot_rows)
display(BOOT_TABLE)

BOOT_TABLE.to_csv(
    OUTDIR/"current_KQSE_bootstrap10_raw.csv",
    index=False
)

In [ ]:
# ================================================================
# 14D. Stability summary and original-run percentile
# ================================================================

BASE_CURRENT = FINAL_TABLE[
    (FINAL_TABLE["Cutoff"]==10)
    & (FINAL_TABLE["Method"]=="KQSE")
][
    ["Observable","ISE","E_inf"]
].copy()

summary_rows = []

for obs in [
    "Wigner","w_0","w_1","w_2","w_3"
]:
    s = BOOT_TABLE[
        BOOT_TABLE["Observable"]==obs
    ]

    base = BASE_CURRENT[
        BASE_CURRENT["Observable"]==obs
    ].iloc[0]

    for metric in ["ISE","E_inf"]:
        vals = np.asarray(
            s[metric],
            float
        )
        base_val = float(
            base[metric]
        )

        summary_rows.append({
            "Observable": obs,
            "Metric": metric,
            "base_original_data": base_val,
            "bootstrap_mean": float(np.mean(vals)),
            "bootstrap_std": float(np.std(vals, ddof=1)),
            "bootstrap_q10": float(np.quantile(vals,0.10)),
            "bootstrap_median": float(np.median(vals)),
            "bootstrap_q90": float(np.quantile(vals,0.90)),
            "CV_std_over_mean": float(
                np.std(vals,ddof=1)
                / np.mean(vals)
            ),
            "base_percentile_among_bootstrap":
                float(np.mean(vals <= base_val)),
        })

BOOT_SUMMARY = pd.DataFrame(summary_rows)
display(BOOT_SUMMARY)

BOOT_SUMMARY.to_csv(
    OUTDIR/"current_KQSE_bootstrap10_summary.csv",
    index=False
)

In [ ]:
# ================================================================
# 15. Does the KQSE ranking look robust without R=100?
# ================================================================

decision_rows = []

for cutoff in [10,20]:
    external = FINAL_TABLE[
        (FINAL_TABLE["Cutoff"]==cutoff)
        & (FINAL_TABLE["Method"]!="KQSE")
    ]

    for obs in [
        "Wigner","w_0","w_1","w_2","w_3"
    ]:
        for metric in ["ISE","E_inf"]:
            boot = BOOT_SUMMARY[
                (BOOT_SUMMARY["Observable"]==obs)
                & (BOOT_SUMMARY["Metric"]==metric)
            ].iloc[0]

            ext = external[
                external["Observable"]==obs
            ]

            best_external = float(
                ext[metric].min()
            )
            best_external_method = str(
                ext.loc[
                    ext[metric].idxmin(),
                    "Method"
                ]
            )

            q10 = float(
                boot["bootstrap_q10"]
            )
            q90 = float(
                boot["bootstrap_q90"]
            )

            if q90 < best_external:
                status = "KQSE advantage robust in pilot"
            elif q10 > best_external:
                status = f"{best_external_method} advantage robust in pilot"
            else:
                status = "pilot bands overlap competitor"

            decision_rows.append({
                "Cutoff": cutoff,
                "Observable": obs,
                "Metric": metric,
                "KQSE_boot_q10": q10,
                "KQSE_boot_q90": q90,
                "best_external_method": best_external_method,
                "best_external_value": best_external,
                "pilot_status": status,
            })

PILOT_DECISION = pd.DataFrame(decision_rows)
display(PILOT_DECISION)

print("\nSummary counts:")
print(
    PILOT_DECISION[
        "pilot_status"
    ].value_counts()
)

overlap_count = int(
    np.sum(
        PILOT_DECISION["pilot_status"]
        == "pilot bands overlap competitor"
    )
)

print("\nDecision aid:")
if overlap_count == 0:
    print(
        "No 10--90% bootstrap band overlaps the best saved competitor. "
        "For the single-dataset head-to-head benchmark, R=100 KQSE-only "
        "repetitions are unlikely to change the qualitative ranking."
    )
else:
    print(
        f"{overlap_count} comparisons have overlapping pilot bands. "
        "Inspect those observables/metrics. If they are central to the paper, "
        "increase the pilot to 20--30 runs before deciding on R=100."
    )

print(
    "\nImportant: a genuine R=100 MISE/MSE_inf claim is a different task. "
    "For a fair R=100 cross-method benchmark, BI/CO/CGAN would also need to "
    "be rerun on the same 100 datasets."
)

## Interpretation rule

For the revised paper, the direct KQSE/BI/CO/CGAN table should remain a
**single-common-dataset comparison**, because the competing methods are only available as
single saved reconstructions.

Use this pilot only to answer:

> Is the current KQSE result on that common dataset obviously atypical?

If the 10-run bootstrap spread is narrow and the qualitative \(N=10\) / \(N=20\) rankings do
not overlap, there is little methodological value in running **100 KQSE-only** repetitions.

Run \(R=100\) only if you want to make a separate statistical-risk statement for KQSE itself,
or if all competing methods will also be rerun on the same 100 independent datasets.